In [ ]:
Sys.time()

In [ ]:
suppressPackageStartupMessages(library(ArchR))
suppressPackageStartupMessages(library(ggplot2))
suppressPackageStartupMessages(library(RColorBrewer))
suppressPackageStartupMessages(library(cowplot))
suppressPackageStartupMessages(library(gridExtra))
suppressPackageStartupMessages(library(ggsci))
suppressPackageStartupMessages(library(scales))
suppressPackageStartupMessages(library(viridis))
suppressPackageStartupMessages(library(tidyverse))
suppressPackageStartupMessages(library(BSgenome))
suppressPackageStartupMessages(library(BSgenome.Hsapiens.UCSC.hg38))

In [ ]:
addArchRGenome("hg38")
addArchRThreads(threads = 8)

In [ ]:
projdir <- '/u/project/cluo/terencew/igvf/2023_YR2/multiome/'
samples <- read.table(paste0(projdir, 'txt/samples.txt'), sep='\t')$V1

In [ ]:
proj_path <- paste0(projdir, 'atac/archr/processed/YS3_intersect_rna')
proj <- loadArchRProject(path = proj_path, force = FALSE, showLogo = TRUE)

In [ ]:
proj

In [ ]:
clust_path = paste0(projdir, 'csv/clusters/rna_leiden_v7.csv')
clusters = read.csv(clust_path, sep='\t', row.names=1)

In [ ]:
head(clusters)

In [ ]:
split_strings <- strsplit(rownames(clusters), "_")
barcodes <- sapply(split_strings, `[`, 1)
sample <- sapply(split_strings, `[`, 2)
clusters$archr_barcode <- paste0(sample, '#', barcodes)
clusters$barcode_sample <- rownames(clusters)
rownames(clusters) <- clusters$archr_barcode

In [ ]:
clusters$coarse_donor <- paste0(clusters$coarse_v7, '_', clusters$assignment)
clusters$cluster_donor <- paste0(clusters$clusters_v7, '_', clusters$assignment)

In [ ]:
proj <- addCellColData(proj,
                       data = clusters$coarse_v7,
                       name='coarse_v7',
                       cells=rownames(clusters))

In [ ]:
peaks_path <- paste0(projdir, '/bed/peaks/intersect_rna_coarse_v7.bed')
peaks <- read.table(peaks_path, sep='\t')
dim(peaks)

In [ ]:
head(peaks)

In [ ]:
tmp_peaks <- peaks[,1:3]
colnames(tmp_peaks) <- c('chrom', 'start', 'end')
peak_set <- GRanges(tmp_peaks)
head(peak_set)

In [ ]:
proj <- addPeakSet(
      ArchRProj = proj,
      peakSet = peak_set,
      genomeAnnotation = getGenomeAnnotation(proj),
      force = FALSE
    )

In [ ]:
proj

In [ ]:
proj <- addPeakMatrix(
      ArchRProj = proj,
      ceiling = 4,
      binarize = FALSE,
      verbose = TRUE,
      threads = getArchRThreads(),
      parallelParam = NULL,
      force = TRUE,
      logFile = createLogFile("addPeakMatrix")
    )

In [ ]:
markerPeaks <- getMarkerFeatures(
    ArchRProj = proj, 
    useMatrix = "PeakMatrix", 
    groupBy = "coarse",
  bias = c("TSSEnrichment", "log10(nFrags)"),
  testMethod = "wilcoxon"
)

In [ ]:
markerPeaks

In [ ]:
outdir = paste0(projdir, '/atac/peaks/archr/cluster')
saveRDS(markerPeaks, paste0(outdir, '/coarse_v7.differential.rds'))

In [ ]:
# markerList <- getMarkers(markerPeaks, cutOff = "FDR <= 0.01 & Log2FC >= 1")

In [ ]:
# sapply(markerList, nrow)

In [ ]:
Sys.time()

In [ ]:
sessionInfo()